In [1]:
import sys

sys.path.append("C:/cloud_backend/langchain_app")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
model = ChatOpenAI(model="gpt-4o-mini")

model.invoke([HumanMessage("안녕?")])

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fc3bf93e48', 'id': 'chatcmpl-EmCHSsTjVQxpgut1fgn2XCoeEWdqD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a1f79d-818d-713c-836f-28ee2b0d08ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 're

In [2]:
from util_tools import get_current_time

# 사용자도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
model_with_tools = model.bind_tools(tools)

In [3]:
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]


response = model_with_tools.invoke(messages)
messages.append(response)


print(messages)

# AI가 tool을 사용하기로 결정하면 바로 사용자에게 응답하지 않고 (AIMessage의 content 항목이 채워지지 않음)
# 대신 tool_calls=[{'name': 'get_current_time', 
#                   'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 
#                   id': 'call_2IA5QMnTGGkgW6PCba9Kt4Qk', 
#                   'type': 'tool_call'}]

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d5265d470', 'id': 'chatcmpl-EmCHWSGDI3YvDfgiJwZCvl9Md2fhf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'normalized_finish_reason': 'tool_call'}, id='lc_run--01a1f79d-9126-72e8-9483-901a2b3293cf-0', tool_calls=[{'name': 'g

In [4]:
# tool_calls 내용을 이용해서 도구 호출
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출(invoke)하여 결과를 반환
    # tool로 지정된 함수를 직접호출 할 수 없다. 반드시 invoke 메서드를 사용해야 함
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-11-20 15:21:02 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d5265d470', 'id': 'chatcmpl-EmCHWSGDI3YvDfgiJwZCvl9Md2fhf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'normalized_finish_reason': 'tool_call'}, id='lc_run--01a1f79d-9126-72e8-9483-901a2b3293cf-0', tool_calls=[{'name': 

In [5]:
# 이렇게 작성된 메시지를 LLM에게 전달
model_with_tools.invoke(messages)

AIMessage(content='부산의 현재 시각은 2026년 11월 20일 15시 21분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 192, 'total_tokens': 218, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d5265d470', 'id': 'chatcmpl-EmCHZLkN4PtX8Nfjev3CZirBbfNEW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a1f79d-9c3b-74e4-a3e3-a6aee50a8c35-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 26, 'total_tokens': 218, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_deta

In [6]:
from util_tools import get_yf_stock_history

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

model_with_tools = model.bind_tools(tools)

In [7]:
messages.append(HumanMessage("마이크론테크놀로지는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = model_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 288, 'total_tokens': 314, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fc3bf93e48', 'id': 'chatcmpl-EmCHc61SJxPcw55jsgQkPUH91LhK6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'normalized_finish_reason': 'tool_call'} id='lc_run--01a1f79d-a970-7770-b9c5-66cc9455fb25-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'MU', 'period': '1mo'}}, 'id': 'call_Pi4HQMCtz5T1xaxKdwBWesaj', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 

In [8]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg.content)

{'stock_history_input': {'ticker': 'MU', 'period': '1mo'}}
| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |
|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|
| 2026-10-20 00:00:00-04:00 | 884.13 | 918.89 | 881.52 |  896.07 | 2.03042e+07 |           0 |              0 |
| 2026-10-21 00:00:00-04:00 | 923.74 | 928.45 | 889.45 |  913.67 | 2.1035e+07  |           0 |              0 |
| 2026-10-22 00:00:00-04:00 | 879.66 | 908.3  | 849.25 |  856.49 | 3.2176e+07  |           0 |              0 |
| 2026-10-23 00:00:00-04:00 | 862.29 | 891.89 | 858.34 |  877.55 | 2.12499e+07 |           0 |              0 |
| 2026-10-26 00:00:00-04:00 | 889.88 | 915.75 | 882.77 |  903.7  | 2.28489e+07 |           0 |              0 |
| 2026-10-27 00:00:00-04:00 | 920.99 | 935.87 | 910.67 |  928.76 | 2.31949e+07 |           0 |              0 |
| 2026-10-28 00:00:00-04:00 | 961.71 | 990   

In [11]:
print(model_with_tools.invoke(messages).content)

마이크론 테크놀로지(Micron Technology, MU)의 주가는 한 달 전인 2026년 10월 20일의 종가 896.07에서 현재(2026년 11월 19일) 종가 690.44로 내려갔습니다. 따라서 한 달 전과 비교했을 때 주가는 하락했습니다.
